<a href="https://colab.research.google.com/github/HarishRock0/DSGP/blob/child-protection-component/script/Phase_2_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: Feature Engineering

## Objective
Transform raw district time-series data into meaningful features that capture:
1. **Current State** (2024 snapshot)
2. **Historical Trends** (2012-2024 patterns)
3. **Comparative Position** (relative to other districts)

## Why Feature Engineering?
Raw data alone doesn't reveal allocation priorities. For example:
- District A: 100 abuse cases in 2024
- District B: 80 abuse cases in 2024

Without feature engineering, District A seems worse. But if:
- District A: Decreasing 10% annually, strong infrastructure
- District B: Increasing 20% annually, poor infrastructure

Then District B actually needs more resources despite lower current cases.

Feature engineering captures these nuanced patterns.

## Inputs
- `combined_districts.csv` from Phase 1

## Outputs
- `district_features.csv`: One row per district with ~35 engineered features
- `feature_descriptions.txt`: Documentation of all features

## 1. Setup and Load Data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.precision', 4)

print("Libraries imported successfully")

Mounted at /content/drive
Libraries imported successfully


In [2]:
# Configuration
INPUT_FILE = '/content/drive/My Drive/output/combined_districts.csv'
OUTPUT_FOLDER = '/content/drive/My Drive/output/'

# Target year for allocation (we'll use 2024 data + trends to allocate for 2025)
CURRENT_YEAR = 2024

print(f"Configuration:")
print(f"  Input file: {INPUT_FILE}")
print(f"  Current year for analysis: {CURRENT_YEAR}")
print(f"  Allocation target: 2025")

Configuration:
  Input file: /content/drive/My Drive/output/combined_districts.csv
  Current year for analysis: 2024
  Allocation target: 2025


In [3]:
# Load cleaned data from Phase 1
df = pd.read_csv(INPUT_FILE)

print(f"Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"Districts: {df['District'].nunique()}")
print(f"Year range: {df['Year'].min()} to {df['Year'].max()}")
print(f"\nDistricts in dataset: {sorted(df['District'].unique())}")

# Display sample
print("\nFirst few rows:")
print(df.head())

Data loaded successfully
Shape: (39, 19)
Districts: 3
Year range: 2012 to 2024

Districts in dataset: ['Colombo', 'Hambantota', 'Matara']

First few rows:
  District  Year  Total population  0-4 Male Child population  \
0  Colombo  2012           2324349                      84185   
1  Colombo  2013           2324349                      84185   
2  Colombo  2014           2343000                      82403   
3  Colombo  2015           2375000                      77287   
4  Colombo  2016           2396773                      82442   

   5-9 Male Child population  10-14 Male Child population  \
0                      87380                        83683   
1                      87380                        83683   
2                      85513                        81315   
3                      80487                        86032   
4                      85704                        86553   

   15-19 Male Child population  0-4 Female Child population  \
0                       

## 2. Current State Features (2024 Snapshot)

These features capture the current situation in each district.

WHY: Resource allocation must address current needs. These features show:
- Scale of population and problems
- Infrastructure adequacy
- Service utilization

In [4]:
def calculate_current_state_features(df, year=2024):
    """
    Calculate current state features for each district.

    WHY: These represent the immediate situation that resource allocation
         must address. They include both raw counts and derived ratios.

    Args:
        df: Combined district dataframe
        year: Year to use as 'current' (default 2024)

    Returns:
        DataFrame with one row per district and current state features
    """
    # Filter to current year
    current_data = df[df['Year'] == year].copy()

    if len(current_data) == 0:
        print(f"WARNING: No data for year {year}. Using most recent year available.")
        latest_year = df['Year'].max()
        current_data = df[df['Year'] == latest_year].copy()
        print(f"Using year {latest_year} instead.")

    features = pd.DataFrame()
    features['District'] = current_data['District']

    # Calculate total child population (0-19 years)
    # WHY: Child population is the denominator for all per-capita calculations
    child_pop_cols = [
        '0-4 Male Child population', '5-9 Male Child population',
        '10-14 Male Child population', '15-19 Male Child population',
        '0-4 Female Child population', '5-9 Female Child population',
        '10-14 Female Child population', '15-19 Female Child population'
    ]
    features['total_child_population'] = current_data[child_pop_cols].sum(axis=1).values

    # Raw counts
    features['total_population'] = current_data['Total population'].values
    features['abuse_cases'] = current_data['Reported chid abuse cases'].values
    features['num_schools'] = current_data['No of schools'].values
    features['num_students'] = current_data['No of students'].values
    features['num_teachers'] = current_data['No of teachers'].values
    features['num_childrens_homes'] = current_data["No of childrens' homes"].values
    features['children_in_homes'] = (
        current_data['No of male children in childrens home'].values +
        current_data["No of female children in childrens' home"].values
    )

    # Critical ratios
    # WHY: Absolute numbers don't show severity - a district with 100 cases
    #      and 10,000 children is different from 100 cases and 50,000 children

    # Abuse rate per 1000 children
    features['abuse_rate_per_1000'] = (
        features['abuse_cases'] / features['total_child_population'] * 1000
    )

    # Student-teacher ratio
    # WHY: High ratios indicate overstretched education system, less supervision
    features['student_teacher_ratio'] = (
        features['num_students'] / features['num_teachers']
    )

    # Schools per 1000 children
    # WHY: Access to education infrastructure
    features['schools_per_1000_children'] = (
        features['num_schools'] / features['total_child_population'] * 1000
    )

    # Children's home capacity utilization
    # WHY: High utilization indicates strained resources
    # Assuming each home has average capacity (we'll use children/homes ratio)
    features['children_per_home'] = (
        features['children_in_homes'] / features['num_childrens_homes'].replace(0, np.nan)
    )

    # Child dependency ratio (children as % of total population)
    # WHY: Higher ratios mean more children to protect with same total resources
    features['child_dependency_ratio'] = (
        features['total_child_population'] / features['total_population'] * 100
    )

    # School enrollment rate (students / total child population)
    # WHY: Lower rates might indicate access issues or dropout
    features['school_enrollment_rate'] = (
        features['num_students'] / features['total_child_population'] * 100
    )

    # Children in institutional care rate
    # WHY: Indicator of family separation and child welfare system load
    features['institutional_care_rate_per_1000'] = (
        features['children_in_homes'] / features['total_child_population'] * 1000
    )

    return features


print("Calculating current state features...")
current_features = calculate_current_state_features(df, CURRENT_YEAR)

print(f"\nCurrent state features calculated: {len(current_features.columns)-1} features")
print(f"Districts: {len(current_features)}")
print("\nFeature preview:")
print(current_features.head())

Calculating current state features...

Current state features calculated: 15 features
Districts: 3

Feature preview:
      District  total_child_population  total_population  abuse_cases  \
12     Colombo                  719417           2459983         1174   
25  Hambantota                  500804            680054          393   
38      Matara                  286323            869032          315   

    num_schools  num_students  num_teachers  num_childrens_homes  \
12          393        325525         17219                   22   
25          320        136902          7847                    4   
38          356        156180          9833                    5   

    children_in_homes  abuse_rate_per_1000  student_teacher_ratio  \
12                480               1.6319                18.9050   
25                 89               0.7847                17.4464   
38                 96               1.1002                15.8833   

    schools_per_1000_children  children_

## 3. Temporal Trend Features (2012-2024 Patterns)

These features capture how situations are changing over time.

WHY: Two districts with identical 2024 numbers can have very different futures:
- Improving districts might need less intervention
- Deteriorating districts need urgent action before crisis
- Accelerating problems require different response than stable situations

In [6]:
def calculate_trend_slope(district_df, column, years_back=None):
    """
    Calculate linear regression slope for a time series.

    WHY: Using linear regression (not just % change) because:
         - Robust to single-year anomalies
         - Gives consistent trend direction
         - Slope magnitude indicates speed of change

    Positive slope = increasing trend
    Negative slope = decreasing trend
    Magnitude = speed of change

    Args:
        district_df: DataFrame for single district
        column: Column name to analyze
        years_back: Number of years to include (None = all years)

    Returns:
        Slope value
    """
    if years_back:
        district_df = district_df.tail(years_back)

    if len(district_df) < 3:
        return 0  # Not enough data for trend

    years = district_df['Year'].values
    values = district_df[column].values

    # Handle missing values
    valid_mask = ~np.isnan(values)
    if valid_mask.sum() < 3:
        return 0

    years_valid = years[valid_mask]
    values_valid = values[valid_mask]

    # Linear regression
    slope, _, _, _, _ = stats.linregress(years_valid, values_valid)

    return slope


def calculate_percent_change(district_df, column, years_back):
    """
    Calculate percentage change over specified period.

    WHY: % change is intuitive and comparable across different scales.
         For example, abuse cases increasing from 50 to 100 (100% change)
         is comparable to population growing from 100k to 200k (100% change)

    Args:
        district_df: DataFrame for single district
        column: Column name to analyze
        years_back: Number of years for comparison

    Returns:
        Percentage change
    """
    recent = district_df.tail(years_back)

    if len(recent) < 2:
        return 0

    start_val = recent.iloc[0][column]
    end_val = recent.iloc[-1][column]

    if pd.isna(start_val) or pd.isna(end_val) or start_val == 0:
        return 0

    pct_change = ((end_val - start_val) / start_val) * 100

    return pct_change


def calculate_volatility(district_df, column):
    """
    Calculate coefficient of variation (CV) as volatility measure.

    WHY: CV = (std / mean) shows relative variability.
         High CV indicates erratic, unpredictable situation.
         Low CV indicates stable, predictable pattern.

         Erratic districts may need different interventions than
         stable districts (e.g., investigation vs routine programs).

    Args:
        district_df: DataFrame for single district
        column: Column name to analyze

    Returns:
        Coefficient of variation
    """
    values = district_df[column].values

    valid_values = values[~np.isnan(values)]

    if len(valid_values) < 3:
        return 0

    mean = np.mean(valid_values)
    std = np.std(valid_values)

    if mean == 0:
        return 0

    cv = (std / mean) * 100

    return cv


def calculate_temporal_features(df):
    """
    Calculate temporal trend features for each district.

    WHY: Capture dynamics over time - essential for proactive allocation.
         Three types of temporal features:
         1. Long-term trends (full 2012-2024 period)
         2. Recent trends (last 3 years) - more weight on recent changes
         3. Volatility - stability vs chaos

    Args:
        df: Combined district dataframe

    Returns:
        DataFrame with temporal features
    """
    districts = df['District'].unique()
    temporal_features = []

    for district in districts:
        district_df = df[df['District'] == district].sort_values('Year')

        features = {'District': district}

        # Abuse cases trends
        # WHY: Most critical metric for child protection
        features['abuse_trend_slope'] = calculate_trend_slope(
            district_df, 'Reported chid abuse cases'
        )
        features['abuse_pct_change_3yr'] = calculate_percent_change(
            district_df, 'Reported chid abuse cases', 3
        )
        features['abuse_pct_change_5yr'] = calculate_percent_change(
            district_df, 'Reported chid abuse cases', 5
        )
        features['abuse_volatility'] = calculate_volatility(
            district_df, 'Reported chid abuse cases'
        )

        # Child population trends
        # WHY: Growing child population means increasing future demand
        child_pop_cols = [
            '0-4 Male Child population', '5-9 Male Child population',
            '10-14 Male Child population', '15-19 Male Child population',
            '0-4 Female Child population', '5-9 Female Child population',
            '10-14 Female Child population', '15-19 Female Child population'
        ]
        district_df['total_child_pop'] = district_df[child_pop_cols].sum(axis=1)

        features['child_pop_trend_slope'] = calculate_trend_slope(
            district_df, 'total_child_pop'
        )
        features['child_pop_pct_change_5yr'] = calculate_percent_change(
            district_df, 'total_child_pop', 5
        )

        # Infrastructure trends
        # WHY: Is infrastructure keeping pace with population?
        features['schools_trend_slope'] = calculate_trend_slope(
            district_df, 'No of schools'
        )
        features['teachers_trend_slope'] = calculate_trend_slope(
            district_df, 'No of teachers'
        )

        # Student-teacher ratio trend
        # WHY: Worsening ratio indicates deteriorating education quality
        district_df['st_ratio'] = district_df['No of students'] / district_df['No of teachers']
        features['st_ratio_trend_slope'] = calculate_trend_slope(
            district_df, 'st_ratio'
        )

        # Children's home trends
        # WHY: Growing institutional care indicates family breakdown
        district_df['total_in_homes'] = (
            district_df['No of male children in childrens home'] +
            district_df["No of female children in childrens' home"]
        )
        features['institutional_care_trend_slope'] = calculate_trend_slope(
            district_df, 'total_in_homes'
        )

        # Acceleration indicators
        # WHY: Is the problem speeding up or slowing down?
        #      Compare recent 3-year trend to overall trend
        recent_slope = calculate_trend_slope(
            district_df, 'Reported chid abuse cases', years_back=3
        )
        overall_slope = features['abuse_trend_slope']

        if overall_slope != 0:
            features['abuse_acceleration'] = recent_slope - overall_slope
        else:
            features['abuse_acceleration'] = recent_slope

        temporal_features.append(features)

    return pd.DataFrame(temporal_features)


print("Calculating temporal trend features...")
print("This may take a moment for trend analysis...\n")

temporal_features = calculate_temporal_features(df)

print(f"Temporal features calculated: {len(temporal_features.columns)-1} features")
print("\nFeature preview:")
print(temporal_features.head())

Calculating temporal trend features...
This may take a moment for trend analysis...

Temporal features calculated: 11 features

Feature preview:
     District  abuse_trend_slope  abuse_pct_change_3yr  abuse_pct_change_5yr  \
0     Colombo             5.3462              -31.2646                3.5273   
1  Hambantota             5.9176               -0.5063                7.3770   
2      Matara            -1.9286              -18.8144                4.6512   

   abuse_volatility  child_pop_trend_slope  child_pop_pct_change_5yr  \
0           19.9528              3966.8187                    0.2018   
1           13.2489             11937.3242                  121.0265   
2           13.4531              1686.1703                    0.2953   

   schools_trend_slope  teachers_trend_slope  st_ratio_trend_slope  \
0              -3.9231               33.6319               -0.3060   
1               0.2527               10.2637                0.0629   
2              -0.5055             

## 4. Comparative Features (Relative Position)

These features show how each district compares to others.

WHY: Absolute values don't show severity in context.
      "100 abuse cases" means different things if:
      - Other districts average 50 (this is high)
      - Other districts average 200 (this is low)

In [7]:
def calculate_comparative_features(current_features):
    """
    Calculate how each district compares to others.

    WHY: Resource allocation is inherently comparative - you're distributing
         a fixed resource pool. Need to know who is worse off relatively.

    Methods used:
    - Percentile ranks: Where does district stand (0-100 scale)
    - Z-scores: How many standard deviations from mean (statistical measure)
    - Gap from best: Distance from best-performing district

    Args:
        current_features: DataFrame with current state features

    Returns:
        DataFrame with comparative features
    """
    comparative = current_features[['District']].copy()

    # Key metrics to compare
    compare_metrics = [
        'abuse_rate_per_1000',
        'student_teacher_ratio',
        'schools_per_1000_children',
        'children_per_home',
        'institutional_care_rate_per_1000'
    ]

    for metric in compare_metrics:
        if metric in current_features.columns:
            values = current_features[metric].values

            # Percentile rank (0-100)
            # WHY: Easy to interpret - 90th percentile means worse than 90% of districts
            # For abuse rate: higher percentile = worse (more abuse)
            # For schools: higher percentile = better (more schools)
            percentile = stats.rankdata(values, method='average') / len(values) * 100
            comparative[f'{metric}_percentile'] = percentile

            # Z-score (standardized score)
            # WHY: Shows how extreme a value is. |z| > 2 is unusual, |z| > 3 is rare
            mean = np.mean(values)
            std = np.std(values)
            if std > 0:
                z_scores = (values - mean) / std
                comparative[f'{metric}_zscore'] = z_scores

            # Gap from best/worst (depending on metric)
            # WHY: Shows how much improvement needed to reach best performance
            if 'abuse' in metric or 'ratio' in metric.lower():
                # Lower is better
                best = np.min(values)
                gap = values - best
                comparative[f'{metric}_gap_from_best'] = gap
            else:
                # Higher is better
                best = np.max(values)
                gap = best - values
                comparative[f'{metric}_gap_from_best'] = gap

    return comparative


print("Calculating comparative features...")
comparative_features = calculate_comparative_features(current_features)

print(f"\nComparative features calculated: {len(comparative_features.columns)-1} features")
print("\nFeature preview:")
print(comparative_features.head())

Calculating comparative features...

Comparative features calculated: 15 features

Feature preview:
      District  abuse_rate_per_1000_percentile  abuse_rate_per_1000_zscore  \
12     Colombo                        100.0000                      1.3148   
25  Hambantota                         33.3333                     -1.1085   
38      Matara                         66.6667                     -0.2062   

    abuse_rate_per_1000_gap_from_best  student_teacher_ratio_percentile  \
12                             0.8471                          100.0000   
25                             0.0000                           66.6667   
38                             0.3154                           33.3333   

    student_teacher_ratio_zscore  student_teacher_ratio_gap_from_best  \
12                        1.2104                               3.0217   
25                        0.0283                               1.5632   
38                       -1.2386                               0.00

## 5. Combine All Features

Merge current state, temporal, and comparative features into single dataset.

In [8]:
# Merge all feature sets
# WHY: Using left joins to ensure no districts are lost

all_features = current_features.copy()

# Add temporal features
all_features = all_features.merge(
    temporal_features,
    on='District',
    how='left'
)

# Add comparative features
all_features = all_features.merge(
    comparative_features,
    on='District',
    how='left'
)

print("="*80)
print("FEATURE ENGINEERING COMPLETE")
print("="*80)
print(f"\nTotal features created: {len(all_features.columns) - 1}")
print(f"Districts: {len(all_features)}")
print(f"\nFeature categories:")
print(f"  - Current state features: {len(current_features.columns) - 1}")
print(f"  - Temporal features: {len(temporal_features.columns) - 1}")
print(f"  - Comparative features: {len(comparative_features.columns) - 1}")

print("\nDataset shape:", all_features.shape)
print("\nColumn names:")
for i, col in enumerate(all_features.columns, 1):
    print(f"{i:2d}. {col}")

FEATURE ENGINEERING COMPLETE

Total features created: 41
Districts: 3

Feature categories:
  - Current state features: 15
  - Temporal features: 11
  - Comparative features: 15

Dataset shape: (3, 42)

Column names:
 1. District
 2. total_child_population
 3. total_population
 4. abuse_cases
 5. num_schools
 6. num_students
 7. num_teachers
 8. num_childrens_homes
 9. children_in_homes
10. abuse_rate_per_1000
11. student_teacher_ratio
12. schools_per_1000_children
13. children_per_home
14. child_dependency_ratio
15. school_enrollment_rate
16. institutional_care_rate_per_1000
17. abuse_trend_slope
18. abuse_pct_change_3yr
19. abuse_pct_change_5yr
20. abuse_volatility
21. child_pop_trend_slope
22. child_pop_pct_change_5yr
23. schools_trend_slope
24. teachers_trend_slope
25. st_ratio_trend_slope
26. institutional_care_trend_slope
27. abuse_acceleration
28. abuse_rate_per_1000_percentile
29. abuse_rate_per_1000_zscore
30. abuse_rate_per_1000_gap_from_best
31. student_teacher_ratio_percenti

## 6. Handle Infinite and Missing Values

Check for any problematic values created during feature engineering.

In [9]:
# Check for infinite values
# WHY: Division by zero or very small numbers can create inf values

print("Checking for problematic values...\n")

# Replace infinite values with NaN
all_features = all_features.replace([np.inf, -np.inf], np.nan)

# Check missing values by column
missing_summary = pd.DataFrame({
    'Column': all_features.columns,
    'Missing_Count': all_features.isnull().sum(),
    'Missing_Percent': (all_features.isnull().sum() / len(all_features) * 100).round(2)
})

missing_summary = missing_summary[missing_summary['Missing_Count'] > 0]

if len(missing_summary) > 0:
    print("Columns with missing values:")
    print(missing_summary.to_string(index=False))
    print("\nFilling missing values with 0...")

    # Fill NaN with 0
    # WHY: In feature engineering, NaN often means "not applicable" or "no change"
    #      For example, 0% change, 0 slope, etc.
    numeric_cols = all_features.select_dtypes(include=[np.number]).columns
    all_features[numeric_cols] = all_features[numeric_cols].fillna(0)

    print("Missing values handled.")
else:
    print("No missing values found. Data is clean.")

# Verify no issues remain
final_missing = all_features.isnull().sum().sum()
final_inf = np.isinf(all_features.select_dtypes(include=[np.number])).sum().sum()

print(f"\nFinal check:")
print(f"  Missing values: {final_missing}")
print(f"  Infinite values: {final_inf}")
print(f"  Status: {'PASS' if final_missing == 0 and final_inf == 0 else 'FAIL'}")

Checking for problematic values...

No missing values found. Data is clean.

Final check:
  Missing values: 0
  Infinite values: 0
  Status: PASS


## 7. Feature Summary Statistics

Understand the distribution and range of engineered features.

In [10]:
print("="*80)
print("FEATURE SUMMARY STATISTICS")
print("="*80)

# Summary statistics for key features
key_features = [
    'abuse_rate_per_1000',
    'student_teacher_ratio',
    'abuse_trend_slope',
    'abuse_pct_change_3yr',
    'child_pop_trend_slope'
]

print("\nKey Feature Statistics:")
for feature in key_features:
    if feature in all_features.columns:
        values = all_features[feature]
        print(f"\n{feature}:")
        print(f"  Mean: {values.mean():.4f}")
        print(f"  Median: {values.median():.4f}")
        print(f"  Std Dev: {values.std():.4f}")
        print(f"  Min: {values.min():.4f}")
        print(f"  Max: {values.max():.4f}")

# Full summary
print("\n" + "="*80)
print("Full Summary Statistics:")
print("="*80)
numeric_features = all_features.select_dtypes(include=[np.number])
print(numeric_features.describe().T)

FEATURE SUMMARY STATISTICS

Key Feature Statistics:

abuse_rate_per_1000:
  Mean: 1.1723
  Median: 1.1002
  Std Dev: 0.4281
  Min: 0.7847
  Max: 1.6319

student_teacher_ratio:
  Mean: 17.4116
  Median: 17.4464
  Std Dev: 1.5112
  Min: 15.8833
  Max: 18.9050

abuse_trend_slope:
  Mean: 3.1117
  Median: 5.3462
  Std Dev: 4.3744
  Min: -1.9286
  Max: 5.9176

abuse_pct_change_3yr:
  Mean: -16.8618
  Median: -18.8144
  Std Dev: 15.4718
  Min: -31.2646
  Max: -0.5063

child_pop_trend_slope:
  Mean: 5863.4377
  Median: 3966.8187
  Std Dev: 5382.3240
  Min: 1686.1703
  Max: 11937.3242

Full Summary Statistics:
                                                count        mean  \
total_child_population                            3.0  5.0218e+05   
total_population                                  3.0  1.3364e+06   
abuse_cases                                       3.0  6.2733e+02   
num_schools                                       3.0  3.5633e+02   
num_students                                 

## 8. Save Engineered Features

Export the feature dataset for use in Phase 3 (EDA) and Phase 4 (Modeling).

In [11]:
import os

# Save feature dataset
output_file = os.path.join(OUTPUT_FOLDER, 'district_features.csv')
all_features.to_csv(output_file, index=False)

print(f"Feature dataset saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024:.2f} KB")

# Create feature documentation
doc_file = os.path.join(OUTPUT_FOLDER, 'feature_descriptions.txt')

with open(doc_file, 'w') as f:
    f.write("FEATURE DOCUMENTATION\n")
    f.write("="*80 + "\n\n")

    f.write("CURRENT STATE FEATURES (2024 Snapshot)\n")
    f.write("-"*80 + "\n")
    f.write("total_child_population: Total children aged 0-19\n")
    f.write("total_population: Total district population\n")
    f.write("abuse_cases: Reported child abuse cases in 2024\n")
    f.write("abuse_rate_per_1000: Abuse cases per 1000 children (severity indicator)\n")
    f.write("student_teacher_ratio: Students per teacher (education capacity)\n")
    f.write("schools_per_1000_children: School access indicator\n")
    f.write("children_per_home: Average children per children's home (capacity strain)\n")
    f.write("child_dependency_ratio: Children as % of total population\n")
    f.write("school_enrollment_rate: Students as % of child population\n")
    f.write("institutional_care_rate_per_1000: Children in homes per 1000 children\n")
    f.write("\n")

    f.write("TEMPORAL TREND FEATURES (2012-2024 Patterns)\n")
    f.write("-"*80 + "\n")
    f.write("abuse_trend_slope: Linear trend in abuse cases (+ = increasing, - = decreasing)\n")
    f.write("abuse_pct_change_3yr: % change in abuse cases over last 3 years\n")
    f.write("abuse_pct_change_5yr: % change in abuse cases over last 5 years\n")
    f.write("abuse_volatility: Coefficient of variation (stability indicator)\n")
    f.write("abuse_acceleration: Recent trend minus overall trend (speeding up/down)\n")
    f.write("child_pop_trend_slope: Trend in child population growth\n")
    f.write("child_pop_pct_change_5yr: % change in child population over 5 years\n")
    f.write("schools_trend_slope: Trend in school infrastructure\n")
    f.write("teachers_trend_slope: Trend in teacher availability\n")
    f.write("st_ratio_trend_slope: Trend in student-teacher ratio\n")
    f.write("institutional_care_trend_slope: Trend in institutional care\n")
    f.write("\n")

    f.write("COMPARATIVE FEATURES (Relative Position)\n")
    f.write("-"*80 + "\n")
    f.write("*_percentile: Rank among districts (0-100, higher = worse for abuse)\n")
    f.write("*_zscore: Standard deviations from mean (|z|>2 is unusual)\n")
    f.write("*_gap_from_best: Distance from best-performing district\n")
    f.write("\n")

    f.write("INTERPRETATION GUIDE\n")
    f.write("-"*80 + "\n")
    f.write("High priority indicators:\n")
    f.write("  - High abuse_rate_per_1000\n")
    f.write("  - Positive abuse_trend_slope (increasing)\n")
    f.write("  - Positive abuse_acceleration (speeding up)\n")
    f.write("  - High student_teacher_ratio\n")
    f.write("  - Positive st_ratio_trend_slope (worsening)\n")
    f.write("  - High abuse_rate_per_1000_percentile (>75th)\n")
    f.write("\n")
    f.write("Lower priority indicators:\n")
    f.write("  - Low abuse_rate_per_1000\n")
    f.write("  - Negative abuse_trend_slope (decreasing)\n")
    f.write("  - Negative abuse_acceleration (slowing down)\n")
    f.write("  - Low student_teacher_ratio\n")
    f.write("  - High schools_per_1000_children\n")

print(f"Feature documentation saved to: {doc_file}")

print("\n" + "="*80)
print("PHASE 2 COMPLETE")
print("="*80)
print("\nOutputs:")
print(f"  1. {output_file}")
print(f"  2. {doc_file}")
print("\nYou can now proceed to Phase 3: Exploratory Data Analysis")

Feature dataset saved to: /content/drive/My Drive/output/district_features.csv
File size: 2.85 KB
Feature documentation saved to: /content/drive/My Drive/output/feature_descriptions.txt

PHASE 2 COMPLETE

Outputs:
  1. /content/drive/My Drive/output/district_features.csv
  2. /content/drive/My Drive/output/feature_descriptions.txt

You can now proceed to Phase 3: Exploratory Data Analysis


## 9. Preview Final Feature Dataset

Display the final engineered features for verification.

In [12]:
print("="*80)
print("FINAL FEATURE DATASET PREVIEW")
print("="*80)

print("\nComplete feature set for all districts:")
print(all_features.to_string())

print("\n" + "="*80)
print("Key insights from features:")
print("="*80)

# Identify districts with concerning trends
print("\nDistricts with INCREASING abuse trends:")
increasing = all_features[all_features['abuse_trend_slope'] > 0].sort_values(
    'abuse_trend_slope', ascending=False
)
if len(increasing) > 0:
    print(increasing[['District', 'abuse_trend_slope', 'abuse_pct_change_3yr']].to_string(index=False))
else:
    print("  None (all districts stable or improving)")

print("\nDistricts with HIGHEST current abuse rates:")
highest_abuse = all_features.nlargest(5, 'abuse_rate_per_1000')
print(highest_abuse[['District', 'abuse_rate_per_1000', 'abuse_cases']].to_string(index=False))

print("\nDistricts with WORST student-teacher ratios:")
worst_st = all_features.nlargest(5, 'student_teacher_ratio')
print(worst_st[['District', 'student_teacher_ratio', 'num_teachers']].to_string(index=False))

print("\n" + "="*80)
print("Features are ready for Phase 3 analysis!")
print("="*80)

FINAL FEATURE DATASET PREVIEW

Complete feature set for all districts:
     District  total_child_population  total_population  abuse_cases  num_schools  num_students  num_teachers  num_childrens_homes  children_in_homes  abuse_rate_per_1000  student_teacher_ratio  schools_per_1000_children  children_per_home  child_dependency_ratio  school_enrollment_rate  institutional_care_rate_per_1000  abuse_trend_slope  abuse_pct_change_3yr  abuse_pct_change_5yr  abuse_volatility  child_pop_trend_slope  child_pop_pct_change_5yr  schools_trend_slope  teachers_trend_slope  st_ratio_trend_slope  institutional_care_trend_slope  abuse_acceleration  abuse_rate_per_1000_percentile  abuse_rate_per_1000_zscore  abuse_rate_per_1000_gap_from_best  student_teacher_ratio_percentile  student_teacher_ratio_zscore  student_teacher_ratio_gap_from_best  schools_per_1000_children_percentile  schools_per_1000_children_zscore  schools_per_1000_children_gap_from_best  children_per_home_percentile  children_per_home_zs